<a href="https://colab.research.google.com/github/kumarsirish/FDP-AGENENTIC-AI-RAG/blob/main/rag-adk-06/google_adk_single_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Google ADK — Single Agent Hands-On Lab

**For professors & students** | Runs entirely on Google Colab

---

## What you'll build

A single AI agent powered by **Google Gemini** that can:
- Look up weather for any city
- Do maths (add, multiply)
- Save and recall notes within the session
- Tell the current time

You'll see **how the agent decides which tool to use**, inspect the raw event stream, and run a multi-turn conversation.

---

## Before you start

1. **Get a free Gemini API key** → [aistudio.google.com](https://aistudio.google.com)
2. **Add it to Colab Secrets**:
   - Click the 🔑 key icon in the left sidebar
   - Add a secret named `GOOGLE_API_KEY` with your key as the value
   - Toggle **Notebook access** ON
3. Run cells **top to bottom** in order.
4. After Cell 1, you must **restart the runtime** (Runtime → Restart session), then continue from Cell 2.

---
## Cell 1 — Install Google ADK

> ⚠️ After running this cell, go to **Runtime → Restart session**, then continue.

In [3]:
!pip install -q google-adk
print("✅ google-adk installed. Now restart the runtime: Runtime → Restart session")

✅ google-adk installed. Now restart the runtime: Runtime → Restart session


---
## Cell 2 — Load API Key

Reads your Gemini API key from Colab Secrets (never hardcode it in the notebook).

In [4]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("✅ API key loaded from Colab Secrets.")

✅ API key loaded from Colab Secrets.


---
## Cell 3 — Import ADK

Import the core ADK components we'll use throughout the lab.

In [5]:
from google.adk.agents import LlmAgent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools import ToolContext
from google.genai.types import Content, Part

print("✅ ADK imports successful.")
print()
print("Key classes loaded:")
print("  LlmAgent           — the AI agent")
print("  InMemorySessionService — stores conversation history")
print("  Runner             — runs agent turn-by-turn")
print("  ToolContext        — lets tools read/write session state")
print("  Content / Part     — message format for Gemini")

✅ ADK imports successful.

Key classes loaded:
  LlmAgent           — the AI agent
  InMemorySessionService — stores conversation history
  Runner             — runs agent turn-by-turn
  ToolContext        — lets tools read/write session state
  Content / Part     — message format for Gemini


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


---
## Cell 4 — Define the Tools

Tools are just **plain Python functions**. ADK reads the docstring and type hints automatically to tell the agent what each tool does and what arguments it expects.

> 📌 **Key insight:** The docstring IS the prompt for the tool. Write it clearly — the agent decides when to call a tool based entirely on reading it.

In [6]:
# ── Tool 1: Weather lookup ─────────────────────────────────────────────────

def get_weather(city: str) -> dict:
    """Get the current weather for a given city.

    Args:
        city: The name of the city to look up (e.g. 'London', 'Tokyo').

    Returns:
        A dict with 'temperature' (Celsius) and 'condition' (str).
    """
    # Mock data — swap this for a real weather API in production
    weather_db = {
        "london":  {"temperature": 15, "condition": "cloudy"},
        "tokyo":   {"temperature": 28, "condition": "sunny"},
        "new york":{"temperature": 22, "condition": "partly cloudy"},
        "paris":   {"temperature": 18, "condition": "rainy"},
        "sydney":  {"temperature": 20, "condition": "clear"},
        "mumbai":  {"temperature": 34, "condition": "humid"},
        "bangalore": {"temperature": 26, "condition": "pleasant"},
    }
    key = city.strip().lower()
    return weather_db.get(key, {"temperature": 20, "condition": "unknown — city not in database"})


# ── Tool 2: Current time ───────────────────────────────────────────────────

def get_current_time() -> str:
    """Get the current date and time.

    Returns:
        The current UTC date and time as a formatted string.
    """
    from datetime import datetime, timezone
    now = datetime.now(timezone.utc)
    return now.strftime("%A, %d %B %Y at %H:%M UTC")


# Show all tools defined
all_tools = [get_weather, get_current_time]
print("✅ Tools defined:")
for t in all_tools:
    print(f"   • {t.__name__}() — {t.__doc__.strip().splitlines()[0]}")

✅ Tools defined:
   • get_weather() — Get the current weather for a given city.
   • get_current_time() — Get the current date and time.


---
## Cell 5 — Create the Agent

We wrap our tools in an `LlmAgent`. The four key fields:

| Field | Purpose |
|---|---|
| `name` | Internal identifier |
| `model` | Which Gemini model to use |
| `instruction` | System prompt — guides the agent's behaviour |
| `tools` | List of Python functions the agent can call |

In [7]:
root_agent = LlmAgent(
    name="smart_assistant",
    model="gemini-2.5-flash",
    description="A helpful assistant with weather, maths, time, and notes tools.",
    instruction=(
        "You are a helpful and concise assistant. "
        "Use your tools whenever a question requires live data or calculation. "
        "For weather, always report both temperature and condition. "
        "For notes, confirm what was saved. "
        "Keep responses brief and friendly."
    ),
    tools=[
        get_weather,
        get_current_time,
    ],
)

print(f"✅ Agent '{root_agent.name}' created.")
print(f"   Model      : {root_agent.model}")
print(f"   Tools ({len(root_agent.tools)}): {[t.__name__ if hasattr(t,'__name__') else str(t) for t in all_tools]}")

✅ Agent 'smart_assistant' created.
   Model      : gemini-2.5-flash
   Tools (2): ['get_weather', 'get_current_time']


---
## Cell 6 — Set Up the Runner & Session

- **`InMemorySessionService`** — keeps conversation history in RAM (resets when Colab disconnects)
- **`Runner`** — orchestrates the turn-by-turn execution loop
- **`Session`** — a unique conversation context (can have many per app)

In [8]:
APP_NAME  = "adk_lab"
USER_ID   = "student_01"

session_service = InMemorySessionService()

runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
)

print("✅ Runner and session ready.")
print(f"   Session ID : {session.id}")
print(f"   App name   : {APP_NAME}")
print(f"   User ID    : {USER_ID}")

✅ Runner and session ready.
   Session ID : 9a1be8b1-ebfd-45f9-b3b2-80fe5c13f325
   App name   : adk_lab
   User ID    : student_01


---
## Cell 7 — Helper: `ask()` function

This helper sends a message and prints only the final response. We'll use it for clean demos.

Below it we also define `ask_verbose()` which prints **every event** including tool calls — great for teaching how the agent reasons step by step.

In [9]:
def ask_verbose(question: str) -> None:
    """Send a question and print every event — tool calls AND final response."""
    print(f"\n{'─'*55}")
    print(f"🧑 User : {question}")
    print(f"{'─'*55}")

    message = Content(role="user", parts=[Part(text=question)])

    for event in runner.run(
        user_id=USER_ID,
        session_id=session.id,
        new_message=message,
    ):
        if not event.content:
            continue
        for part in event.content.parts:
            # Tool call — agent is invoking a function
            if hasattr(part, "function_call") and part.function_call:
                fc = part.function_call
                print(f"  🔧 [tool call ] {fc.name}({dict(fc.args)})")
            # Tool result — the function's return value
            elif hasattr(part, "function_response") and part.function_response:
                fr = part.function_response
                print(f"  📦 [tool result] {fr.name} → {fr.response}")
            # Final text response
            elif hasattr(part, "text") and part.text and event.is_final_response():
                print(f"🤖 Agent: {part.text.strip()}")


print("✅ ask() and ask_verbose() helpers ready.")

✅ ask() and ask_verbose() helpers ready.


---
## Cell 8 — Demo: Weather Tool

Ask about the weather. The agent will call `get_weather()` automatically.

In [10]:
ask_verbose("What's the weather like in Bangalore right now?")


───────────────────────────────────────────────────────
🧑 User : What's the weather like in Bangalore right now?
───────────────────────────────────────────────────────


  🔧 [tool call ] get_weather({'city': 'Bangalore'})
  📦 [tool result] get_weather → {'temperature': 26, 'condition': 'pleasant'}
🤖 Agent: The weather in Bangalore is pleasant with a temperature of 26 degrees Celsius.


---
## Cell 9 — Demo: Multi-Tool Question

A single question that requires multiple tool calls. Watch the agent chain them together.

In [13]:
ask_verbose("What is the weather in London and Paris? Also what time is it now?")


───────────────────────────────────────────────────────
🧑 User : What is the weather in London and Paris? Also what time is it now?
───────────────────────────────────────────────────────
  🔧 [tool call ] get_weather({'city': 'London'})
  🔧 [tool call ] get_weather({'city': 'Paris'})
  🔧 [tool call ] get_current_time({})
  📦 [tool result] get_weather → {'temperature': 15, 'condition': 'cloudy'}
  📦 [tool result] get_weather → {'temperature': 18, 'condition': 'rainy'}
  📦 [tool result] get_current_time → {'result': 'Wednesday, 10 June 2026 at 09:15 UTC'}
🤖 Agent: In London, it's cloudy with a temperature of 15 degrees Celsius. In Paris, it's rainy with a temperature of 18 degrees Celsius. The current time is Wednesday, 10 June 2026 at 09:15 UTC.


In [15]:
ask_verbose("which cities?")


───────────────────────────────────────────────────────
🧑 User : which cities?
───────────────────────────────────────────────────────
🤖 Agent: London, Paris, and Bangalore.
